In [7]:
import osmnx as ox
import pandas as pd
import geopandas as gpd


##using osmnx to get example table with cinemas

place_name = "Berlin, Germany"
tags = {
    "shop": ["bakery", "pastry"]
}

# This works in OSMnx 2.x
gdf = ox.features_from_place(place_name, tags)
gdf.head()


geometry addr:city addr:country  \
element id                                                            
node    58489996   POINT (13.40829 52.51035)    Berlin           DE   
        253999302  POINT (13.35618 52.48628)    Berlin           DE   
        254333065  POINT (13.32023 52.49613)       NaN          NaN   
        254992626  POINT (13.26299 52.48796)       NaN          NaN   
        258050855  POINT (13.29729 52.50696)    Berlin           DE   

                  addr:housenumber addr:postcode       addr:street  \
element id                                                           
node    58489996                77         10179  Alte Jakobstraße   
        253999302               22         10827       Hauptstraße   
        254333065              NaN           NaN               NaN   
        254992626              NaN           NaN               NaN   
        258050855               80         10627        Kantstraße   

                      addr:suburb    shop bakehouse         brand  ... room  \
element id                                                         ...        
node    58489996            Mitte  bakery       NaN           NaN  ...  NaN   
        253999302      Schöneberg  bakery        no  Back-Factory  ...  NaN   
        254333065             NaN  bakery       NaN           NaN  ...  NaN   
        254992626             NaN  bakery       NaN     Steinecke  ...  NaN   
        258050855  Charlottenburg  bakery       NaN           NaN  ...  NaN   

                  roof:levels office building:part covered nohousenumber  \
element id                                                                 
node    58489996          NaN    NaN           NaN     NaN           NaN   
        253999302         NaN    NaN           NaN     NaN           NaN   
        254333065         NaN    NaN           NaN     NaN           NaN   
        254992626         NaN    NaN           NaN     NaN           NaN   
        258050855         NaN    NaN           NaN     NaN           NaN   

                  coffee location shop_1 description:de  
element id                                               
node    58489996     NaN      NaN    NaN            NaN  
        253999302    NaN      NaN    NaN            NaN  
        254333065    NaN      NaN    NaN            NaN  
        254992626    NaN      NaN    NaN            NaN  
        258050855    NaN      NaN    NaN            NaN  

[5 rows x 199 columns]

In [15]:
#example of selecting needed columns
columns = ['name', 'shop', 'geometry', 'addr:street', 'addr:housenumber', 'addr:postcode', 'website', 'operator']
bakery = gdf[columns]
bakery.head()

name    shop                   geometry  \
element id                                                              
node    58489996               NaN  bakery  POINT (13.40829 52.51035)   
        253999302     Back-Factory  bakery  POINT (13.35618 52.48628)   
        254333065  Brot & Brötchen  bakery  POINT (13.32023 52.49613)   
        254992626        Steinecke  bakery  POINT (13.26299 52.48796)   
        258050855     Wilmina Brot  bakery  POINT (13.29729 52.50696)   

                        addr:street addr:housenumber addr:postcode website  \
element id                                                                   
node    58489996   Alte Jakobstraße               77         10179     NaN   
        253999302       Hauptstraße               22         10827     NaN   
        254333065               NaN              NaN           NaN     NaN   
        254992626               NaN              NaN           NaN     NaN   
        258050855        Kantstraße               80         10627     NaN   

                  operator  
element id                  
node    58489996       NaN  
        253999302      NaN  
        254333065      NaN  
        254992626      NaN  
        258050855      NaN

In [11]:
# Load official Berlin districts GeoDataFrame from lor_ortsteile.geojson
berlin_districts_gdf = gpd.read_file("/Users/nigar_kauser/Documents/Webeet_internship/Bakeries/layered-populate-data-pool-da/bakeries/sources/lor_ortsteile.geojson")

In [16]:
# Spatial join: matching your bakery with district and neighborhoods
bakery_with_districts = gpd.sjoin(
    bakery,
    berlin_districts_gdf[["BEZIRK", "spatial_name", "OTEIL","geometry"]],
    how="left",
    predicate="within"
)
bakery_with_districts.head()

name    shop                   geometry  \
element id                                                              
node    58489996               NaN  bakery  POINT (13.40829 52.51035)   
        253999302     Back-Factory  bakery  POINT (13.35618 52.48628)   
        254333065  Brot & Brötchen  bakery  POINT (13.32023 52.49613)   
        254992626        Steinecke  bakery  POINT (13.26299 52.48796)   
        258050855     Wilmina Brot  bakery  POINT (13.29729 52.50696)   

                        addr:street addr:housenumber addr:postcode website  \
element id                                                                   
node    58489996   Alte Jakobstraße               77         10179     NaN   
        253999302       Hauptstraße               22         10827     NaN   
        254333065               NaN              NaN           NaN     NaN   
        254992626               NaN              NaN           NaN     NaN   
        258050855        Kantstraße               80         10627     NaN   

                  operator  index_right                      BEZIRK  \
element id                                                            
node    58489996       NaN            0                       Mitte   
        253999302      NaN           44        Tempelhof-Schöneberg   
        254333065      NaN           22  Charlottenburg-Wilmersdorf   
        254992626      NaN           24  Charlottenburg-Wilmersdorf   
        258050855      NaN           21  Charlottenburg-Wilmersdorf   

                  spatial_name           OTEIL  
element id                                      
node    58489996          0101           Mitte  
        253999302         0701      Schöneberg  
        254333065         0402     Wilmersdorf  
        254992626         0404       Grunewald  
        258050855         0401  Charlottenburg

In [17]:
##just renaming columns for proper schema
bakery_with_districts = bakery_with_districts.rename(columns={
    "BEZIRK": "district",
    "OTEIL": "neighborhood",
    "spatial_name": "neighborhood_id"
}).drop(columns=["index_right"])  # drop district_number if not needed
bakery_with_districts.head()

name    shop                   geometry  \
element id                                                              
node    58489996               NaN  bakery  POINT (13.40829 52.51035)   
        253999302     Back-Factory  bakery  POINT (13.35618 52.48628)   
        254333065  Brot & Brötchen  bakery  POINT (13.32023 52.49613)   
        254992626        Steinecke  bakery  POINT (13.26299 52.48796)   
        258050855     Wilmina Brot  bakery  POINT (13.29729 52.50696)   

                        addr:street addr:housenumber addr:postcode website  \
element id                                                                   
node    58489996   Alte Jakobstraße               77         10179     NaN   
        253999302       Hauptstraße               22         10827     NaN   
        254333065               NaN              NaN           NaN     NaN   
        254992626               NaN              NaN           NaN     NaN   
        258050855        Kantstraße               80         10627     NaN   

                  operator                    district neighborhood_id  \
element id                                                               
node    58489996       NaN                       Mitte            0101   
        253999302      NaN        Tempelhof-Schöneberg            0701   
        254333065      NaN  Charlottenburg-Wilmersdorf            0402   
        254992626      NaN  Charlottenburg-Wilmersdorf            0404   
        258050855      NaN  Charlottenburg-Wilmersdorf            0401   

                     neighborhood  
element id                         
node    58489996            Mitte  
        253999302      Schöneberg  
        254333065     Wilmersdorf  
        254992626       Grunewald  
        258050855  Charlottenburg

In [20]:
# District mapping (official codes as strings)
district_mapping = {
    'Mitte': '11001001',
    'Friedrichshain-Kreuzberg': '11002002',
    'Pankow': '11003003',
    'Charlottenburg-Wilmersdorf': '11004004',
    'Spandau': '11005005',
    'Steglitz-Zehlendorf': '11006006',
    'Tempelhof-Schöneberg': '11007007',
    'Neukölln': '11008008',
    'Treptow-Köpenick': '11009009',
    'Marzahn-Hellersdorf': '11010010',
    'Lichtenberg': '11011011',
    'Reinickendorf': '11012012'
}

# Apply mapping to create district_id column (string)
bakery_with_districts['district_id'] = bakery_with_districts['district'].map(district_mapping).astype(str)

# (Optional) Check if some districts were not mapped
#unmapped = bakery_with_districts[~bakery_with_districts['district'].isin(district_mapping.keys())]['district'].unique()
#if len(unmapped) > 0:
#    print("⚠️ Unmapped districts found:", unmapped)
bakery_with_districts.head()

name    shop                   geometry  \
element id                                                              
node    58489996               NaN  bakery  POINT (13.40829 52.51035)   
        253999302     Back-Factory  bakery  POINT (13.35618 52.48628)   
        254333065  Brot & Brötchen  bakery  POINT (13.32023 52.49613)   
        254992626        Steinecke  bakery  POINT (13.26299 52.48796)   
        258050855     Wilmina Brot  bakery  POINT (13.29729 52.50696)   

                        addr:street addr:housenumber addr:postcode website  \
element id                                                                   
node    58489996   Alte Jakobstraße               77         10179     NaN   
        253999302       Hauptstraße               22         10827     NaN   
        254333065               NaN              NaN           NaN     NaN   
        254992626               NaN              NaN           NaN     NaN   
        258050855        Kantstraße               80         10627     NaN   

                  operator                    district neighborhood_id  \
element id                                                               
node    58489996       NaN                       Mitte            0101   
        253999302      NaN        Tempelhof-Schöneberg            0701   
        254333065      NaN  Charlottenburg-Wilmersdorf            0402   
        254992626      NaN  Charlottenburg-Wilmersdorf            0404   
        258050855      NaN  Charlottenburg-Wilmersdorf            0401   

                     neighborhood district_id  
element id                                     
node    58489996            Mitte    11001001  
        253999302      Schöneberg    11007007  
        254333065     Wilmersdorf    11004004  
        254992626       Grunewald    11004004  
        258050855  Charlottenburg    11004004